# Лабораторная работа 4.

### Воробьев Андрей 465440 J3211
### Шакина Анна 396675 J3211

# Задание № 1

1. Подберите «словесный» датасет
2. Предоработайте выбранный датасет (уберите «ненужные»/«бессмысленные» токены)
3. Постройте переходную матрицу, где в качестве состояний выступают слова
4. Исследуйте адекватность полученной модели (можно самым наивным способом: генерация фраз и подсчёт количества «адекватных»)
5. *Дополнительно (неоцениваемое как основная оценка, максимум как бонус):* сравнить результаты с другим NLP-подходом


### Пункт 1.
В качестве датасета возьмем произведение А. С. Пушкина "Золотая рыбка". Выбор обусловлен тем, что:
1. Произведение достаточно короткое, что позволит более наглядно продемонстрировать работу модели, не работая с огромными разреженными матрицами.
2. Произведение написано на простом русском языке, что также позволяет избежать редких слов и сложных конструкций, которые будут встречаться лишь один раз, создавая разреженную матрицу.
3. Произведение содержит множество повторяющихся слов и фраз, что позволяет модели лучше обучаться на этих данных и создавать более адекватные генерации.

Скачаем архив с текстом по ссылке: https://royallib.com/get/txt/pushkin_aleksandr/skazka_o_ribake_i_ribke.zip

И победим проблемы с кодировкой...

In [81]:
from pathlib import Path

import requests
import zipfile

url = "https://royallib.com/get/txt/pushkin_aleksandr/skazka_o_ribake_i_ribke.zip"
zip_path = Path("skazka.zip")
dest = Path("skazka_text")
dest.mkdir(parents=True, exist_ok=True)

zip_path.write_bytes(requests.get(url).content)

with zipfile.ZipFile(zip_path) as zf:
    for zi in zf.infolist():
        name = zi.filename.encode("cp437").decode("cp866")
        data = zf.read(zi).decode("cp1251")
        path = dest / name
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(data, encoding="utf-8")

Откроем файл "Пушкин Александр. Сказка о рыбаке и рыбке - royallib.ru.txt". Удалим всё до первой строки "Жил старик со своею старухой" и удалим всё после строки "А пред нею разбитое корыто.".


In [82]:
with open(dest / "Пушкин Александр. Сказка о рыбаке и рыбке - royallib.ru.txt", "r", encoding="utf-8") as f:
    text = f.read()

In [83]:
start = text.find("Жил старик со своею старухой")
end = text.find("А пред нею разбитое корыто.")
text = text[start:end + len("А пред нею разбитое корыто.")]

In [84]:
print(text)

Жил старик со своею старухой

		У самого синего моря;

		Они жили в ветхой землянке

		Ровно тридцать лет и три года.

		Старик ловил неводом рыбу,

		Старуха пряла свою пряжу.

		Раз он в море закинул невод, —

		Пришёл невод с одною тиной.

		Он в другой раз закинул невод, —

		Пришёл невод с травой морскою.

		В третий раз закинул он невод, —

		Пришёл невод с одною рыбкой.

		С непростою рыбкой, — золотою.

		Как взмолится золотая рыбка!

		Голосом молвит человечьим:

		«Отпусти ты, старче, меня в море,

		Дорогой за себя дам откуп:

		Откуплюсь чем только пожелаешь».

		Удивился старик, испугался:

		Он рыбачил тридцать лет и три года

		И не слыхивал, чтоб рыба говорила.

		Отпустил он рыбку золотую

		И сказал ей ласковое слово:

		«Бог с тобою, золотая рыбка!

		Твоего мне откупа не надо;

		Ступай себе в синее море,

		Гуляй там себе на просторе».

		Воротился старик ко старухе,

		Рассказал ей великое чудо.

		«Я сегодня поймал было рыбку,

		Золотую рыбку, не простую;

		По-

### Пункт 2.
Удалим лишние пробелы, удалим знаки препинания, кроме точек, заменим точки на EOS, все слова переведем в нижний регистр и преобразуем их в начальную форму. Для этого воспользуемся библиотекой natasha.

In [85]:
import re
from natasha import Segmenter, MorphVocab, NewsEmbedding, NewsMorphTagger, Doc

segmenter = Segmenter()
vocab = MorphVocab()
emb = NewsEmbedding()
morph_tagger = NewsMorphTagger(emb)


def preprocess(text):
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^\w\s\.]", "", text)
    text = text.replace(".", "eos")

    doc = Doc(text)
    doc.segment(segmenter)
    doc.tag_morph(morph_tagger)

    tokens = []
    for token in doc.tokens:
        if token.text.strip():
            if token.text == "eos":
                tokens.append("<EOS>")
            else:
                token.lemmatize(vocab)
                tokens.append(token.lemma)

    return tokens

In [86]:
tokens = preprocess(text)

In [87]:
print(text[:100])
print()
print(tokens[:20])

Жил старик со своею старухой

		У самого синего моря;

		Они жили в ветхой землянке

		Ровно тридцат

['жить', 'старик', 'с', 'свой', 'старуха', 'у', 'сам', 'синий', 'море', 'они', 'жить', 'в', 'ветхий', 'землянка', 'ровный', 'тридцать', 'год', 'и', 'три', 'год']


In [88]:
vocab = list(set(tokens))
vocab_size = len(vocab)
word2idx = {word: idx for idx, word in enumerate(vocab)}
idx2word = {idx: word for word, idx in word2idx.items()}

print(f"Всего токенов: {len(tokens)}")
print(f"Размер словаря: {vocab_size}")

Всего токенов: 999
Размер словаря: 318


### Пункт 3.
Построим переходную матрицу. Для этого создадим матрицу размером vocab_size x vocab_size, где элемент (i, j) будет содержать количество переходов от слова с индексом i к слову с индексом j. Затем нормируем эту матрицу, чтобы получить вероятности переходов.

In [89]:
import numpy as np

transition_matrix = np.zeros((vocab_size, vocab_size), dtype=np.float32)

for i in range(len(tokens) - 1):
    current_word = tokens[i]
    next_word = tokens[i + 1]
    current_idx = word2idx[current_word]
    next_idx = word2idx[next_word]
    transition_matrix[current_idx, next_idx] += 1

row_sums = transition_matrix.sum(axis=1, keepdims=True)
transition_matrix = np.divide(
    transition_matrix,
    row_sums,
    out=np.zeros_like(transition_matrix),
    where=row_sums != 0
)

Выведем столбец переходов от слова "старик, рыбак, золотой, рыбка".

In [90]:
for word in ["старик", "золотой", "рыбка"]:
    idx = word2idx[word]
    print(f"Переходы от слова '{word}':")
    next_word_list = []
    for j in range(vocab_size):
        if transition_matrix[idx, j] > 0:
            next_word_list.append((idx2word[j], transition_matrix[idx, j]))
    next_word_list.sort(key=lambda x: x[1], reverse=True)
    for next_word, prob in next_word_list:
        print(f"  {next_word}: {prob:.4f}")
    print()

Переходы от слова 'старик':
  с: 0.1667
  к: 0.1667
  я: 0.1250
  старуха: 0.0833
  отвечать: 0.0833
  испугаться: 0.0833
  посылать: 0.0417
  взашея: 0.0417
  свой: 0.0417
  не: 0.0417
  привести: 0.0417
  ловить: 0.0417
  взмолиться: 0.0417

Переходы от слова 'золотой':
  рыбка: 0.7500
  и: 0.1250
  перстень: 0.0625
  <EOS>: 0.0625

Переходы от слова 'рыбка':
  не: 0.2121
  <EOS>: 0.1515
  спросить: 0.1212
  золотой: 0.0606
  приплыть: 0.0606
  голос: 0.0303
  и: 0.0303
  хоть: 0.0303
  еще: 0.0303
  старик: 0.0303
  что: 0.0303
  пуще: 0.0303
  поклониться: 0.0303
  домой: 0.0303
  разбранить: 0.0303
  лишь: 0.0303
  опять: 0.0303
  твой: 0.0303



### Пункт 4.
Напишем функцию для генерации фраз на основе построенной модели. Функция будет принимать начальное слово и генерировать фразу, выбирая следующие слова на основе вероятностей переходов. Генерация будет продолжаться, пока не встретится <EOS> или не будет достигнута максимальная длина фразы.

In [91]:
import random


def generate_phrase(matrix, start_words, word2idx, idx2word, max_length=20):
    current_word = random.choice([w for w in start_words if w != '<EOS>'])
    phrase = [current_word]

    for _ in range(max_length - 1):
        current_idx = word2idx[current_word]
        probabilities = matrix[current_idx]

        next_idx = np.random.choice(vocab_size, p=probabilities)
        next_word = idx2word[next_idx]

        if next_word == '<EOS>':
            phrase.append(next_word)
            break

        phrase.append(next_word)
        current_word = next_word

    return " ".join(phrase) + "."

In [92]:
N_phrases = 5
generated_phrases = []

print("Сгенерированные фразы:")
for i in range(N_phrases):
    phrase = generate_phrase(transition_matrix, ["старик", "золотой", "старуха"], word2idx, idx2word)
    generated_phrases.append(phrase)
    print(f"{i + 1}. {phrase}")

Сгенерированные фразы:
1. старуха не печалиться ступать себя дать откуп откупиться что свет стоить он к синий море закинуть невод рыба старуха за.
2. старуха воротиться старик с поклон отвечать золотой <EOS>.
3. золотой рыбка спросить чего ты сам она стоить грозный стража на крыльцо стоить он насмеяться поделом ты невежа впредь ты.
4. старик старуха забранить дурачина ты надобно старча она в море не хотеть быть бы у я покой изба вы новый.
5. старик старуха бунтовать уж быть столбовой дворянка <EOS>.


Сгенерированные фразы:

    1. _старик отвечать золотой и быть вы уж изба <EOS>._
    2. _золотой рыбка хоть бы у старуха по щека ударить муж <EOS>._
    3. _старуха сидеть он велеть <EOS>._
    4. _золотой и ходить так и сказать рыбка не спокойно синий море помутилось синий море гулять там себя дать откуп не._
    5. _старик отвечать смиловаться государыня рыбка не печалиться ступать себя с трава морской <EOS>._

Все фразы имеют похожую структуру на фразы из оригинального текста. Возникает характерая для такого подхода проблема повторения слов и зацикливания, в связи с отсутствием памяти у модели. Тем не менее, многие фразы выглядят адекватными и напоминают стиль оригинального текста, что говорит о возможности применения данного подхода для получения более чем 0 адекватных фраз.

### Пункт 5.
Для сравнения с другим NLP-подходом, мы можем использовать простую модель на основе LSTM. Эта модель будет обучаться на тех же данных и будет использоваться для генерации фраз. Для чистоты сравнения построим очень маленькую LSTM с числом параметров в 2 раза меньше, чем в нашей матрице переходов. Это позволит избежать переобучения и даст более честное сравнение с моделью на основе переходной матрицы.

In [97]:
import torch.nn as nn

class TinyLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=16, hidden_dim=64):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)

        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, dropout=0.2)

        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden=None):
        x = self.embed(x)
        out, hidden = self.lstm(x, hidden)

        out = self.fc(out)
        return out, hidden

model = TinyLSTM(vocab_size=vocab_size)

params = sum(p.numel() for p in model.parameters())
print(f"Параметров в LSTM: {params} (против {vocab_size * vocab_size} в матрице переходов)")

Параметров в LSTM: 46750 (против 101124 в матрице переходов)


In [98]:
import torch
from torch.utils.data import Dataset, DataLoader

class TextDataset(Dataset):
    def __init__(self, tokens, word2idx, seq_len=5):
        self.seq_len = seq_len
        self.data = [word2idx[t] for t in tokens]

    def __len__(self):
        return len(self.data) - self.seq_len

    def __getitem__(self, idx):
        x = torch.LongTensor(self.data[idx : idx + self.seq_len])
        y = torch.tensor(self.data[idx + self.seq_len], dtype=torch.long)
        return x, y

SEQ_LEN = 6
BATCH_SIZE = 16

dataset = TextDataset(tokens, word2idx, seq_len=SEQ_LEN)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

In [99]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 100
model.train()

for epoch in range(EPOCHS):
    total_loss = 0
    for x_batch, y_batch in dataloader:
        optimizer.zero_grad()

        y_pred, _ = model(x_batch)
        last_step_logits = y_pred[:, -1, :]

        loss = criterion(last_step_logits, y_batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if (epoch + 1) % 5 == 0:
        print(f"Эпоха {epoch+1}/{EPOCHS}, Loss: {total_loss/len(dataloader):.4f}")

Эпоха 5/100, Loss: 4.7275
Эпоха 10/100, Loss: 3.7183
Эпоха 15/100, Loss: 2.8241
Эпоха 20/100, Loss: 2.0684
Эпоха 25/100, Loss: 1.5284
Эпоха 30/100, Loss: 1.1056
Эпоха 35/100, Loss: 0.7971
Эпоха 40/100, Loss: 0.5878
Эпоха 45/100, Loss: 0.4199
Эпоха 50/100, Loss: 0.3116
Эпоха 55/100, Loss: 0.2349
Эпоха 60/100, Loss: 0.1861
Эпоха 65/100, Loss: 0.1483
Эпоха 70/100, Loss: 0.1220
Эпоха 75/100, Loss: 0.1294
Эпоха 80/100, Loss: 0.0879
Эпоха 85/100, Loss: 0.0752
Эпоха 90/100, Loss: 0.0702
Эпоха 95/100, Loss: 0.0606
Эпоха 100/100, Loss: 0.0602


In [101]:
def generate_lstm_text(model, start_phrase, word2idx, idx2word, max_len=20, temp=1.0):
    model.eval()
    words = start_phrase.lower().split()
    input_indices = [word2idx.get(w, random.choice(list(word2idx.values()))) for w in words[-SEQ_LEN:]]

    result = list(words)

    with torch.no_grad():
        for _ in range(max_len):
            x = torch.LongTensor([input_indices]).to(next(model.parameters()).device)
            logits, _ = model(x)

            probs = torch.softmax(logits[0, -1, :] / temp, dim=-1)

            next_idx = torch.multinomial(probs, 1).item()
            next_word = idx2word[next_idx]

            if next_word == '<EOS>':
                result.append('<EOS>')
                break

            result.append(next_word)
            input_indices = input_indices[1:] + [next_idx]

    return " ".join(result) + "."

seed = "старик отвечать золотой"
print("\nРезультат генерации LSTM:")
for i in range(N_phrases):
    generated = generate_lstm_text(model, seed, word2idx, idx2word)
    print(f"{i + 1}. {generated}")


Результат генерации LSTM:
1. старик отвечать золотой свой рыбка не хотеть быть она крестьянка хотеть быть вольный царица хотеть быть владычица так я рыбку не жить свой.
2. старик отвечать золотой свой глубокий <EOS>.
3. старик отвечать золотой рыбка <EOS>.
4. старик отвечать золотой рыбка спросить рыбка поклониться она теперь золотой поклонися рыбка <EOS>.
5. старик отвечать золотой надобно рыбка не хотеть быть столбовой дворянка <EOS>.


Результат генерации LSTM:

    1. _старик отвечать золотой свой рыбка не хотеть быть она крестьянка хотеть быть вольный царица хотеть быть владычица так я рыбку не жить свой._
    2. _старик отвечать золотой свой глубокий <EOS>._
    3. _старик отвечать золотой рыбка <EOS>._
    4. _старик отвечать золотой рыбка спросить рыбка поклониться она теперь золотой поклонися рыбка <EOS>._
    5. _старик отвечать золотой надобно рыбка не хотеть быть столбовой дворянка <EOS>._

Генерация LSTM выглядит более разнообразной и менее зацикленной по сравнению с моделью на основе переходной матрицы. Модель LSTM может учитывать контекст и создавать более сложные фразы, в то время как модель на основе переходной матрицы часто повторяет одни и те же слова и фразы. Например в первом предложении LSTM генерирует разнообразную фразу с несмотря одинаковую конструкцию перечисления. В целом LSTM демонстрирует более высокую адекватность, разнообразие и соблюдение контекста при этом храня меньше параметров, а также обучаясь на том же объеме данных, что и модель на основе переходной матрицы. Это говорит о том, что LSTM является более мощным инструментом для генерации текста по сравнению с простыми моделями на основе переходных матриц.